# 📒 Notebook 5 — Evaluation & Results
## Metrics · Plots · Comparison Table

This notebook generates the complete evaluation suite for your thesis:

| Output | Description |
|--------|-------------|
| Confusion matrices | Per-model, color-coded |
| Training history | Loss & F1 curves per model |
| Combined ROC curves | All models on one plot |
| Metric bar chart | Side-by-side comparison |
| Comparison table | CSV + printed summary |
| Classification reports | Precision/Recall/F1 per class |

> You can run this notebook independently by loading saved JSON results from `outputs/`.

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import roc_curve, auc
warnings.filterwarnings('ignore')

# ── Config ────────────────────────────────────────────────
OUTPUT_DIR = "outputs"; LOG_DIR = "logs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
plt.style.use("seaborn-v0_8-whitegrid")

PALETTE = {
    "BERT":       "#1f77b4",
    "RoBERTa":    "#ff7f0e",
    "DistilBERT": "#2ca02c",
    "BERT-CNN":   "#d62728",   # red = proposed model
}
print("Evaluation notebook ready.")

## 5.1 Load Results
Results are loaded from the JSON files saved during training. If you ran Notebook 4 in the same session, `all_results` is already in memory — just run the cell below to also load from disk for robustness.

In [ ]:
def load_results(model_keys=None):
    """Load test results from JSON files saved by the trainer."""
    if model_keys is None:
        model_keys = ["BERT","RoBERTa","DistilBERT","BERT-CNN"]
    results = {}
    for key in model_keys:
        path = os.path.join(OUTPUT_DIR, f"results_{key}.json")
        if os.path.exists(path):
            with open(path) as f: results[key] = json.load(f)
            print(f"  Loaded: {path}")
        else:
            print(f"  ⚠  Not found: {path}  (train this model first)")
    return results

def load_histories(model_keys=None):
    if model_keys is None:
        model_keys = ["BERT","RoBERTa","DistilBERT","BERT-CNN"]
    histories = {}
    for key in model_keys:
        path = os.path.join(LOG_DIR, f"{key}_history.json")
        if os.path.exists(path):
            with open(path) as f: histories[key] = json.load(f)
    return histories

all_results   = load_results()
all_histories = load_histories()
print(f"\nModels available: {list(all_results.keys())}")

## 5.2 Confusion Matrices

In [ ]:
def plot_confusion_matrix(cm, model_name, save_dir=OUTPUT_DIR):
    cm_arr = np.array(cm)
    fig, ax = plt.subplots(figsize=(5,4))
    sns.heatmap(cm_arr, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Non-Suicide","Suicide"],
                yticklabels=["Non-Suicide","Suicide"],
                ax=ax, linewidths=0.5, annot_kws={"size":13})
    ax.set_xlabel("Predicted", fontsize=11); ax.set_ylabel("True", fontsize=11)
    ax.set_title(f"Confusion Matrix — {model_name}", fontweight="bold", fontsize=12)
    plt.tight_layout()
    path = os.path.join(save_dir, f"cm_{model_name}.png")
    plt.savefig(path, dpi=150); plt.show(); print(f"  Saved: {path}")

for name, m in all_results.items():
    if "cm" in m:
        plot_confusion_matrix(m["cm"], name)

## 5.3 Training History Curves

In [ ]:
def plot_training_history(history, model_name, save_dir=OUTPUT_DIR):
    if not history: return
    epochs = range(1, len(history["train"])+1)
    color  = PALETTE.get(model_name,"#333")
    fig, axes = plt.subplots(1,2,figsize=(12,4))
    for ax, metric, label in zip(axes,["loss","f1"],["Loss","Macro F1"]):
        tr = [h[metric] for h in history["train"]]
        vl = [h[metric] for h in history["val"]]
        ax.plot(epochs, tr, "o-", label="Train", color=color, linewidth=2)
        ax.plot(epochs, vl, "s--",label="Val",   color=color, linewidth=2, alpha=0.65)
        ax.set_xlabel("Epoch",fontsize=11); ax.set_ylabel(label,fontsize=11)
        ax.set_title(f"{model_name} — {label}", fontweight="bold")
        ax.legend()
    fig.suptitle(f"Training History: {model_name}", fontsize=13, fontweight="bold")
    plt.tight_layout()
    path = os.path.join(save_dir,f"history_{model_name}.png")
    plt.savefig(path,dpi=150); plt.show(); print(f"  Saved: {path}")

for name, hist in all_histories.items():
    plot_training_history(hist, name)

## 5.4 Combined ROC Curves
All four models on one plot. The proposed BERT-CNN is shown with a dashed red line for emphasis.

> **Note:** ROC-AUC data is stored in results JSON. For the full ROC curve (FPR/TPR arrays), rerun inference using the code cell below.

In [ ]:
# ── Option A: plot from stored AUC scores (simple bar) ───
fig, ax = plt.subplots(figsize=(7,5))
ax.plot([0,1],[0,1],"k--",lw=1.2,label="Random (AUC=0.50)")

roc_placeholder = {}
for name, m in all_results.items():
    auc_val = m.get("roc_auc", 0)
    color   = PALETTE.get(name,"#555")
    lw      = 2.5 if name=="BERT-CNN" else 1.8
    ls      = "--" if name=="BERT-CNN" else "-"
    # Draw a stylized diagonal line (actual FPR/TPR needs re-inference)
    ax.plot([0,1],[0,auc_val*1.05],[],lw=0, label=f"{name}  (AUC = {auc_val:.4f})",
            color=color)  # legend entry only

# Redraw legend with correct styles
handles = []
import matplotlib.lines as mlines
for name, m in all_results.items():
    auc_val = m.get("roc_auc",0)
    h = mlines.Line2D([],[],color=PALETTE.get(name,"#555"),
                      lw=2.5 if name=="BERT-CNN" else 1.8,
                      linestyle="--" if name=="BERT-CNN" else "-",
                      label=f"{name}  (AUC = {auc_val:.4f})")
    handles.append(h)
ax.legend(handles=handles, loc="lower right", fontsize=10)
ax.set_xlabel("False Positive Rate",fontsize=12)
ax.set_ylabel("True Positive Rate",fontsize=12)
ax.set_title("ROC-AUC Comparison — All Models", fontsize=13, fontweight="bold")
plt.tight_layout()
path = os.path.join(OUTPUT_DIR,"roc_auc_comparison.png")
plt.savefig(path,dpi=150); plt.show(); print(f"Saved: {path}")

In [ ]:
# ── Option B: Full ROC curves via re-inference ─────────────
# Uncomment and run this cell if you have model checkpoints saved
# Requires: models, dataloaders from Notebook 4

# import torch
# import torch.nn as nn
# from sklearn.metrics import roc_curve, auc
#
# CHECKPOINT_DIR = "checkpoints"
# roc_data = {}
#
# for model_key in all_results.keys():
#     ckpt = os.path.join(CHECKPOINT_DIR, f"{model_key}_best.pt")
#     if not os.path.exists(ckpt):
#         print(f"Checkpoint missing: {ckpt}"); continue
#     model = get_model(model_key)  # from NB4
#     model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
#     model.eval()
#     all_probs, all_labels = [], []
#     train_loader, val_loader, test_loader = get_dataloaders(MODELS[model_key])
#     with torch.no_grad():
#         for batch in test_loader:
#             logits = model(batch["input_ids"].to(DEVICE),
#                            batch["attention_mask"].to(DEVICE),
#                            batch["token_type_ids"].to(DEVICE))
#             all_probs  += torch.softmax(logits,-1)[:,1].cpu().tolist()
#             all_labels += batch["label"].tolist()
#     fpr,tpr,_ = roc_curve(all_labels, all_probs)
#     roc_data[model_key] = (fpr, tpr, auc(fpr,tpr))
#     del model; torch.cuda.empty_cache()
#
# # Plot
# fig,ax = plt.subplots(figsize=(7,6))
# ax.plot([0,1],[0,1],'k--',lw=1.2,label='Random')
# for name,(fpr,tpr,roc_auc) in roc_data.items():
#     ax.plot(fpr,tpr,lw=2.5 if name=='BERT-CNN' else 1.8,
#             color=PALETTE[name],linestyle='--' if name=='BERT-CNN' else '-',
#             label=f"{name} (AUC={roc_auc:.4f})")
# ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
# ax.set_title("ROC Curves — All Models",fontweight='bold'); ax.legend()
# plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/roc_curves_full.png",dpi=150); plt.show()
print("Uncomment the cell above after training to generate full ROC curves.")

## 5.5 Metric Comparison Bar Chart

In [ ]:
def plot_metric_comparison(results, save_dir=OUTPUT_DIR):
    metrics      = ["accuracy","precision","recall","f1","f1_suicide","roc_auc"]
    metric_labels= ["Accuracy","Precision","Recall","Macro F1","F1 (Suicide)","ROC-AUC"]
    model_names  = list(results.keys())
    x            = np.arange(len(metrics))
    width        = 0.18
    offsets      = np.linspace(-(len(model_names)-1)/2,(len(model_names)-1)/2,len(model_names))*width

    fig, ax = plt.subplots(figsize=(14,6))
    for name, offset in zip(model_names, offsets):
        vals = [results[name].get(m,0) for m in metrics]
        bars = ax.bar(x+offset, vals, width, label=name,
                      color=PALETTE.get(name,"gray"), edgecolor="white", linewidth=0.5)
        for bar, v in zip(bars,vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                    f"{v:.3f}", ha="center", va="bottom", fontsize=6.5, rotation=45)

    ax.set_xticks(x); ax.set_xticklabels(metric_labels, fontsize=11)
    ax.set_ylim(0.7,1.03); ax.set_ylabel("Score",fontsize=12)
    ax.set_title("Model Comparison — All Metrics",fontsize=13,fontweight="bold")
    ax.legend(loc="lower right",fontsize=10)
    plt.tight_layout()
    path = os.path.join(save_dir,"metric_comparison.png")
    plt.savefig(path,dpi=150); plt.show(); print(f"Saved: {path}")

plot_metric_comparison(all_results)

## 5.6 Final Comparison Table

In [ ]:
def generate_comparison_table(results, save_dir=OUTPUT_DIR):
    rows = []
    for name, m in results.items():
        rows.append({
            "Model":         name,
            "Accuracy":      round(m.get("accuracy",0),4),
            "Precision":     round(m.get("precision",0),4),
            "Recall":        round(m.get("recall",0),4),
            "Macro F1":      round(m.get("f1",0),4),
            "F1 (Suicide)":  round(m.get("f1_suicide",0),4),
            "ROC-AUC":       round(m.get("roc_auc",0),4),
        })
    df = pd.DataFrame(rows).set_index("Model")

    # Highlight best value per column
    styled = df.style.highlight_max(axis=0, props="background-color:#d4edda; color:#155724; font-weight:bold")

    print("="*70)
    print("  FINAL COMPARISON TABLE")
    print("="*70)
    print(df.to_string())
    print("="*70)

    # Print improvement of BERT-CNN
    if "BERT-CNN" in results and len(results)>1:
        baselines = {k:v for k,v in results.items() if k!="BERT-CNN"}
        best_f1   = max(v["f1"] for v in baselines.values())
        ours_f1   = results["BERT-CNN"]["f1"]
        delta     = (ours_f1-best_f1)*100
        print(f"\n  BERT-CNN improvement over best baseline — Macro F1: {delta:+.2f} pp")

    path = os.path.join(save_dir,"comparison_table.csv")
    df.to_csv(path); print(f"\nSaved: {path}")
    return styled

generate_comparison_table(all_results)

## 5.7 Classification Reports

In [ ]:
# Classification reports are stored as strings in results JSON
# Print them here for thesis inclusion
for name, m in all_results.items():
    report = m.get("report","Not available")
    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(report)

    # Save to text file
    path = os.path.join(OUTPUT_DIR, f"report_{name}.txt")
    with open(path,"w") as f:
        f.write(f"Classification Report — {name}\n{'='*50}\n{report}")
    print(f"  Saved: {path}")

## 5.8 Summary Statistics for Thesis Write-up

In [ ]:
print("\n" + "="*60)
print("  KEY NUMBERS FOR YOUR THESIS")
print("="*60)

for name, m in all_results.items():
    tag = " ◄ PROPOSED" if name=="BERT-CNN" else ""
    print(f"\n  {name}{tag}")
    print(f"    Accuracy   : {m.get('accuracy',0):.4f}  ({m.get('accuracy',0)*100:.2f}%)")
    print(f"    Macro F1   : {m.get('f1',0):.4f}  ({m.get('f1',0)*100:.2f}%)")
    print(f"    F1-Suicide : {m.get('f1_suicide',0):.4f}  ({m.get('f1_suicide',0)*100:.2f}%)")
    print(f"    ROC-AUC    : {m.get('roc_auc',0):.4f}")

if "BERT-CNN" in all_results and len(all_results)>1:
    baselines = {k:v for k,v in all_results.items() if k!="BERT-CNN"}
    best_base_f1  = max(v["f1"] for v in baselines.values())
    best_base_name= max(baselines, key=lambda k: baselines[k]["f1"])
    ours_f1       = all_results["BERT-CNN"]["f1"]
    print(f"\n  Best baseline: {best_base_name} (F1 = {best_base_f1:.4f})")
    print(f"  BERT-CNN      : F1 = {ours_f1:.4f}")
    print(f"  Improvement   : {(ours_f1-best_base_f1)*100:+.2f} percentage points")

print("\n  All figures saved to: ./outputs/")